# 03 — Transform Silver

## Objective

Transform Bronze raw tables into cleaned, standardized and validated Silver tables.

## This notebook performs

- Reading of Bronze Delta tables.
- Normalization of categorical values.
- Conversion of data types.
- Validation of critical business rules.
- Separation of critical rejected records and warning-level issues into `silver_rejected_records`.
- Creation of cleaned Silver tables.
- Storage of Silver tables as Delta tables.

## Notes

The Silver layer applies business cleaning and validation rules. Correctable issues are standardized, critical invalid records are rejected, and warning-level issues are preserved with flags for later review.

In [0]:
from datetime import datetime
from functools import reduce

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Use the project schema where Bronze tables were created.

spark.sql("USE SCHEMA workbridge")

DataFrame[]

In [0]:
# Read Bronze tables created in the previous notebook.
# These tables still contain raw data plus Bronze metadata.

bronze_headcount_df = spark.table("bronze_headcount")
bronze_clients_df = spark.table("bronze_clients")
bronze_assignments_df = spark.table("bronze_assignments")
bronze_hours_df = spark.table("bronze_hours")
bronze_costs_df = spark.table("bronze_costs")
bronze_goals_df = spark.table("bronze_goals")

In [0]:
# ============================================================
# Normalization functions
# ============================================================
# These functions standardize inconsistent values from raw sources.
# They are used to clean categorical fields before creating Silver tables.
# ============================================================

def normalize_region(column):
    """
    Normalize region values into standard region identifiers:
    LATAM, NA, EMEA, APAC.
    """

    normalized = F.lower(F.trim(column))

    return (
        F.when(normalized.isin("latam", "latin america"), "LATAM")
         .when(normalized.isin("na", "north america"), "NA")
         .when(normalized.isin("emea", "europe", "europe middle east africa"), "EMEA")
         .when(normalized.isin("apac", "asia pacific", "asia-pacific"), "APAC")
         .otherwise(None)
    )


def normalize_gender(column):
    """
    Normalize gender values into:
    Female, Male, Not specified.
    """

    normalized = F.lower(F.trim(column))

    return (
        F.when(normalized.isin("female", "f"), "Female")
         .when(normalized.isin("male", "m"), "Male")
         .when(
             normalized.isin("not specified", "unknown", "n/a", "") | normalized.isNull(),
             "Not specified"
         )
         .otherwise("Not specified")
    )


def normalize_employment_status(column):
    """
    Normalize employment status values into:
    active, terminated.
    Invalid or unknown values return null.
    """

    normalized = F.lower(F.trim(column))

    return (
        F.when(normalized == "active", "active")
         .when(normalized == "terminated", "terminated")
         .otherwise(None)
    )


def normalize_assignment_status(column):
    """
    Normalize assignment status values into:
    active, ended.
    """

    normalized = F.lower(F.trim(column))

    return (
        F.when(normalized == "active", "active")
         .when(normalized.isin("ended", "closed"), "ended")
         .otherwise(None)
    )


def normalize_currency(column):
    """
    Normalize currency values into USD.
    """

    normalized = F.lower(F.trim(column))

    return (
        F.when(normalized.isin("usd", "u$d", "us dollars"), "USD")
         .otherwise(None)
    )


def normalize_contract_type(column):
    """
    Normalize contract type values.
    """

    normalized = F.lower(F.trim(column))

    return (
        F.when(normalized.isin("fixed price", "fp"), "Fixed Price")
         .when(normalized.isin("time and materials", "t&m"), "Time and Materials")
         .when(normalized.isin("managed service", "managed services", "ms"), "Managed Service")
         .otherwise(None)
    )


def normalize_industry(column):
    """
    Normalize industry values.
    """

    normalized = F.lower(F.trim(column))

    return (
        F.when(normalized.isin("technology", "tech", "it"), "Technology")
         .when(normalized.isin("healthcare", "health care"), "Healthcare")
         .when(normalized.isin("banking", "financial services"), "Banking")
         .when(normalized == "telecom", "Telecommunications")
         .otherwise(F.initcap(F.trim(column)))
    )

In [0]:
# ============================================================
# Rejected records helper functions
# ============================================================
# Critical invalid records should not continue into Silver tables.
# Instead, they are stored in a rejected records table with the reason.
# ============================================================

def create_rejected_records(df, source_table, record_id_column, condition, reason):
    """
    Create a rejected records DataFrame for records that meet a rejection condition.

    Parameters:
    - df: source DataFrame.
    - source_table: name of the source Bronze table.
    - record_id_column: column used to identify the rejected record.
    - condition: Spark condition that identifies invalid records.
    - reason: explanation of why the record was rejected.
    """

    return (
        df
        .filter(condition)
        .select(
            F.lit(source_table).alias("source_table"),
            F.col(record_id_column).cast("string").alias("record_id"),
            F.lit(reason).alias("rejection_reason"),
            F.lit("critical").alias("rejection_severity"),
            F.current_timestamp().alias("rejected_at"),
            F.col("bronze_load_id").cast("string").alias("bronze_load_id")
        )
    )


def union_dataframes(dataframes):
    """
    Union multiple DataFrames with the same schema.
    If the list is empty, return None.
    """

    if len(dataframes) == 0:
        return None

    return reduce(lambda df1, df2: df1.unionByName(df2), dataframes)

# Transform Employees

## Source and target

Source table: `bronze_headcount`  
Target table: `silver_employees`

## Transformation rules

- Normalize gender values.
- Normalize region values into `region_id`.
- Normalize employment status values.
- Convert `hire_date` and `termination_date` to date type.
- Validate employee email format using an `email_is_valid` flag.
- Detect duplicated `employee_id` values.
- Distinguish exact duplicated records from conflicting duplicated records.
- Keep one version of exact duplicates in `silver_employees`.
- Exclude conflicting duplicated records from `silver_employees`.
- Register critical invalid records and duplicate-related warnings for traceability.

## Rejection and warning rules

Records are registered as critical when:

- `employee_id` is missing.
- `employment_status` cannot be normalized.
- `termination_date` is earlier than `hire_date`.
- The same `employee_id` appears with conflicting employee information.

Records are registered as warnings when:

- The same `employee_id` appears more than once with identical business information.

## Notes

Invalid emails are not rejected because they do not prevent workforce KPI calculation. Instead, they are preserved in Silver with `email_is_valid = false`.

Exact duplicated employee records are deduplicated by keeping one record in `silver_employees` and logging the duplicate occurrence as a warning.

Conflicting duplicated employee records are excluded from `silver_employees` because the pipeline cannot automatically determine which version is correct

Employees with non-normalizable region values are kept in `silver_employees` with `region_id = null` and monitored later as a warning-level quality indicator.

In [0]:

employees_base_df = (
    bronze_headcount_df
    .withColumn("employee_id", F.trim(F.col("employee_id")))
    .withColumn("employee_name", F.trim(F.col("employee_name")))
    .withColumn("employee_email", F.trim(F.col("employee_email")))
    .withColumn("gender", normalize_gender(F.col("gender")))
    .withColumn("region_id", normalize_region(F.col("region")))
    .withColumn("employment_status", normalize_employment_status(F.col("employment_status")))
    .withColumn("hire_date", F.to_date(F.col("hire_date")))
    .withColumn("termination_date", F.to_date(F.col("termination_date")))
    .withColumn(
        "email_is_valid",
        F.col("employee_email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
    )
)

# Critical rejection conditions.
# These records should not continue into the Silver employees table.

employees_invalid_id_condition = (
    F.col("employee_id").isNull()
    | (F.col("employee_id") == "")
)

employees_invalid_status_condition = (
    F.col("employment_status").isNull()
)

employees_invalid_dates_condition = (
    F.col("termination_date").isNotNull()
    & F.col("hire_date").isNotNull()
    & (F.col("termination_date") < F.col("hire_date"))
)

# Create rejected records for critical employee issues.

employees_critical_rejections = [
    create_rejected_records(
        employees_base_df,
        source_table="bronze_headcount",
        record_id_column="employee_id",
        condition=employees_invalid_id_condition,
        reason="employee_id is missing"
    ),
    create_rejected_records(
        employees_base_df,
        source_table="bronze_headcount",
        record_id_column="employee_id",
        condition=employees_invalid_status_condition,
        reason="employment_status is invalid"
    ),
    create_rejected_records(
        employees_base_df,
        source_table="bronze_headcount",
        record_id_column="employee_id",
        condition=employees_invalid_dates_condition,
        reason="termination_date is earlier than hire_date"
    ),
]

employees_critical_rejected_df = union_dataframes(employees_critical_rejections)

# Keep only records that do not have basic critical issues.

employees_valid_df = (
    employees_base_df
    .filter(~employees_invalid_id_condition)
    .filter(~employees_invalid_status_condition)
    .filter(~employees_invalid_dates_condition)
)

# Detect employee_id values that appear more than once.
# employee_id is expected to uniquely identify an employee.

employee_duplicate_ids_df = (
    employees_valid_df
    .groupBy("employee_id")
    .count()
    .filter(F.col("count") > 1)
    .select("employee_id")
)

employees_duplicate_rows_df = (
    employees_valid_df
    .join(employee_duplicate_ids_df, on="employee_id", how="inner")
)

# To distinguish exact duplicates from conflicting duplicates, we count how many
# different business versions exist for each duplicated employee_id.
#
# If distinct_versions = 1:
#   The duplicated rows are identical in the relevant business columns.
#   We keep one row and log the issue as a warning.
#
# If distinct_versions > 1:
#   The same employee_id appears with conflicting employee information.
#   We reject those records as critical because the pipeline cannot decide
#   which version is correct.

employee_business_columns = [
    "employee_name",
    "employee_email",
    "gender",
    "country",
    "region_id",
    "department",
    "role",
    "seniority",
    "hire_date",
    "termination_date",
    "employment_status",
]

employee_duplicate_versions_df = (
    employees_duplicate_rows_df
    .select(["employee_id"] + employee_business_columns)
    .distinct()
    .groupBy("employee_id")
    .count()
    .withColumnRenamed("count", "distinct_versions")
)

employee_exact_duplicate_ids_df = (
    employee_duplicate_versions_df
    .filter(F.col("distinct_versions") == 1)
    .select("employee_id")
)

employee_conflicting_duplicate_ids_df = (
    employee_duplicate_versions_df
    .filter(F.col("distinct_versions") > 1)
    .select("employee_id")
)

# Log exact duplicated employee_id records as warnings.
# These are not critical because all business values are consistent.
# One version will be kept in silver_employees_df.

employees_exact_duplicates_warning_df = (
    employees_duplicate_rows_df
    .join(employee_exact_duplicate_ids_df, on="employee_id", how="inner")
    .select(
        F.lit("bronze_headcount").alias("source_table"),
        F.col("employee_id").cast("string").alias("record_id"),
        F.lit("exact duplicated employee_id").alias("rejection_reason"),
        F.lit("warning").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Log conflicting duplicated employee_id records as critical.
# These records are excluded from silver_employees_df because the same ID
# appears with different employee information.

employees_conflicting_duplicates_rejected_df = (
    employees_duplicate_rows_df
    .join(employee_conflicting_duplicate_ids_df, on="employee_id", how="inner")
    .select(
        F.lit("bronze_headcount").alias("source_table"),
        F.col("employee_id").cast("string").alias("record_id"),
        F.lit("conflicting employee_id records").alias("rejection_reason"),
        F.lit("critical").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Combine all employee rejected/warning records.

employees_rejected_df = union_dataframes([
    employees_critical_rejected_df,
    employees_exact_duplicates_warning_df,
    employees_conflicting_duplicates_rejected_df
])

# Exclude conflicting duplicated IDs from the Silver employees table.
# Exact duplicates are allowed to continue, but only one record per employee_id is kept.

employees_valid_for_silver_df = (
    employees_valid_df
    .join(employee_conflicting_duplicate_ids_df, on="employee_id", how="left_anti")
)

silver_employees_df = (
    employees_valid_for_silver_df
    .dropDuplicates(["employee_id"])
    .select(
        "employee_id",
        "employee_name",
        "employee_email",
        "email_is_valid",
        "gender",
        "country",
        "region_id",
        "department",
        "role",
        "seniority",
        "hire_date",
        "termination_date",
        "employment_status",
        "source_file",
        "ingestion_timestamp",
        "bronze_load_id"
    )
)

display(silver_employees_df.limit(10))

employee_id,employee_name,employee_email,email_is_valid,gender,country,region_id,department,role,seniority,hire_date,termination_date,employment_status,source_file,ingestion_timestamp,bronze_load_id
E0011,James Moore,james.moore.e0011@workbridge.com,true,Female,United Kingdom,EMEA,HR,Data Analyst,Junior,2024-05-26,null,active,headcount.csv,2026-05-09T12:44:03.800Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
E0026,Ava Thomas,ava.thomas.e0026@workbridge.com,true,Not specified,Canada,NA,HR,Data Engineer,Senior,2023-03-20,2025-10-24,terminated,headcount.csv,2026-05-09T12:44:03.800Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
E0053,James Perez,james.perez.e0053@workbridge.com,true,Female,Argentina,LATAM,Finance,Data Analyst,Junior,2026-03-21,null,active,headcount.csv,2026-05-09T12:44:03.800Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
E0079,Isabella Johnson,isabella.johnson.e0079@workbridge.com,true,Not specified,United States,NA,Finance,Data Analyst,Senior,2026-06-25,null,active,headcount.csv,2026-05-09T12:44:03.800Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
E0136,Daniel Sanchez,daniel.sanchez.e0136@workbridge.com,true,Not specified,United States,NA,Operations,HR Specialist,Lead,2023-02-23,null,active,headcount.csv,2026-05-09T12:44:03.800Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
E0167,Michael Ruiz,michael.ruiz.e0167@workbridge.com,true,Male,Chile,LATAM,Operations,Data Engineer,Junior,2024-08-05,null,active,headcount.csv,2026-05-09T12:44:03.800Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
E0177,John Anderson,john.anderson.e0177@workbridge.com,true,Male,Chile,LATAM,Finance,Data Analyst,Semi Senior,2025-04-17,null,active,headcount.csv,2026-05-09T12:44:03.800Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
E0225,Michael Ruiz,michael.ruiz.e0225@workbridge.com,true,Female,Canada,null,Operations,Business Analyst,Senior,2026-03-04,null,active,headcount.csv,2026-05-09T12:44:03.800Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
E0226,Mia Davis,mia.davis.e0226@workbridge.com,true,Male,Mexico,LATAM,Operations,Software Engineer,Junior,2026-01-23,null,active,headcount.csv,2026-05-09T12:44:03.800Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
E0229,Isabella Lopez,isabella.lopez.e0229@workbridge.com,true,Female,Brazil,LATAM,Engineering,HR Specialist,Lead,2025-04-22,null,active,headcount.csv,2026-05-09T12:44:03.800Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


In [0]:
# Check the result of the employee transformation.
# This quick validation compares the number of records from Bronze,
# the number of valid records kept in Silver, and the number of
# critical or warning records logged during the transformation.

bronze_headcount_count = bronze_headcount_df.count()
silver_employees_count = silver_employees_df.count()

print(f"Bronze headcount rows: {bronze_headcount_count}")
print(f"Silver employees rows: {silver_employees_count}")

if employees_rejected_df is not None:
    rejected_employee_count = employees_rejected_df.count()
    print(f"Rejected or warning employee records: {rejected_employee_count}")
    display(employees_rejected_df)
else:
    print("Rejected or warning employee records: 0")

Bronze headcount rows: 508
Silver employees rows: 493
Rejected or warning employee records: 22


source_table,record_id,rejection_reason,rejection_severity,rejected_at,bronze_load_id
bronze_headcount,E0305,employment_status is invalid,critical,2026-05-09T12:46:09.198Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0429,employment_status is invalid,critical,2026-05-09T12:46:09.198Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0479,employment_status is invalid,critical,2026-05-09T12:46:09.198Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0017,termination_date is earlier than hire_date,critical,2026-05-09T12:46:09.198Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0271,termination_date is earlier than hire_date,critical,2026-05-09T12:46:09.198Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0442,termination_date is earlier than hire_date,critical,2026-05-09T12:46:09.198Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0078,exact duplicated employee_id,warning,2026-05-09T12:46:09.198Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0229,exact duplicated employee_id,warning,2026-05-09T12:46:09.198Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0303,exact duplicated employee_id,warning,2026-05-09T12:46:09.198Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0310,exact duplicated employee_id,warning,2026-05-09T12:46:09.198Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


# Transform Clients

## Source and target

Source table: `bronze_clients`  
Target table: `silver_clients`

## Transformation rules

- Trim text fields.
- Normalize client region values into `region_id`.
- Normalize contract type values.
- Normalize industry values.
- Create a standardized client name for duplicate detection.
- Identify missing account manager values.
- Detect potential duplicate client names.
- Detect duplicated `client_id` values.
- Separate critical invalid records from valid Silver records.

## Critical rejection rules

Records are registered as critical and excluded from `silver_clients` when:

- `client_id` is missing.
- `client_name` is missing.
- `region_id` cannot be normalized.
- `contract_type` cannot be normalized.
- The same `client_id` appears with conflicting client information.

## Warning rules

Records are registered as warnings but preserved in `silver_clients` when:

- `account_manager` is missing.
- A client name appears to be potentially duplicated based on its standardized name.
- The same `client_id` appears more than once with identical business information.

## Notes

Potential duplicate clients by name are not rejected automatically because business confirmation is required before merging client records.

Instead, they are:

- preserved in `silver_clients`;
- flagged with `potential_duplicate_flag = true`;
- registered as warning records for later review.

Missing account managers are also preserved in `silver_clients`, but flagged with `account_manager_missing_flag = true`.

Exact duplicated `client_id` records are deduplicated by keeping one record in `silver_clients` and logging the duplicate occurrence as a warning.

Conflicting duplicated `client_id` records are excluded from `silver_clients` because the pipeline cannot automatically determine which version is correct.

In [0]:
# Transform client data from Bronze to Silver.
# This section normalizes client fields, detects critical invalid records,
# detects exact and conflicting duplicated client IDs, identifies warning-level issues,
# and creates the cleaned silver_clients_df.

clients_base_df = (
    bronze_clients_df
    .withColumn("client_id", F.trim(F.col("client_id")))
    .withColumn("client_name", F.trim(F.col("client_name")))
    .withColumn("industry", normalize_industry(F.col("industry")))
    .withColumn("region_id", normalize_region(F.col("client_region")))
    .withColumn("account_manager", F.trim(F.col("account_manager")))
    .withColumn("contract_type", normalize_contract_type(F.col("contract_type")))
)

# Create a standardized client name for duplicate detection.
# This does not replace the original client name.
# It only creates a comparable version by:
# - converting text to lowercase;
# - removing common company suffixes;
# - removing spaces and punctuation.
#
# Examples:
# "Nova Retail"       -> "novaretail"
# "NovaRetail"        -> "novaretail"
# "Alpha Bank Ltd."   -> "alphabank"

clients_base_df = (
    clients_base_df
    .withColumn(
        "standardized_client_name",
        F.lower(F.trim(F.col("client_name")))
    )
    .withColumn(
        "standardized_client_name",
        F.regexp_replace(F.col("standardized_client_name"), r"\bltd\b|\binc\b|\bllc\b", "")
    )
    .withColumn(
        "standardized_client_name",
        F.regexp_replace(F.col("standardized_client_name"), r"[^a-z0-9]", "")
    )
)

# Critical rejection conditions.
# These records should not continue into the Silver clients table.

clients_invalid_id_condition = (
    F.col("client_id").isNull()
    | (F.col("client_id") == "")
)

clients_invalid_name_condition = (
    F.col("client_name").isNull()
    | (F.col("client_name") == "")
)

clients_invalid_region_condition = (
    F.col("region_id").isNull()
)

clients_invalid_contract_type_condition = (
    F.col("contract_type").isNull()
)

# Create rejected records for basic critical client issues.

clients_basic_critical_rejections = [
    create_rejected_records(
        clients_base_df,
        source_table="bronze_clients",
        record_id_column="client_id",
        condition=clients_invalid_id_condition,
        reason="client_id is missing"
    ),
    create_rejected_records(
        clients_base_df,
        source_table="bronze_clients",
        record_id_column="client_id",
        condition=clients_invalid_name_condition,
        reason="client_name is missing"
    ),
    create_rejected_records(
        clients_base_df,
        source_table="bronze_clients",
        record_id_column="client_id",
        condition=clients_invalid_region_condition,
        reason="client_region could not be normalized"
    ),
    create_rejected_records(
        clients_base_df,
        source_table="bronze_clients",
        record_id_column="client_id",
        condition=clients_invalid_contract_type_condition,
        reason="contract_type could not be normalized"
    ),
]

clients_basic_critical_rejected_df = union_dataframes(clients_basic_critical_rejections)

# Keep only clients that do not have basic critical issues.

clients_valid_df = (
    clients_base_df
    .filter(~clients_invalid_id_condition)
    .filter(~clients_invalid_name_condition)
    .filter(~clients_invalid_region_condition)
    .filter(~clients_invalid_contract_type_condition)
)

# Detect client_id values that appear more than once.
# client_id is expected to uniquely identify a client.

client_duplicate_ids_df = (
    clients_valid_df
    .groupBy("client_id")
    .count()
    .filter(F.col("count") > 1)
    .select("client_id")
)

clients_duplicate_rows_df = (
    clients_valid_df
    .join(client_duplicate_ids_df, on="client_id", how="inner")
)

# Distinguish exact duplicated client IDs from conflicting duplicated client IDs.
#
# If distinct_versions = 1:
#   The duplicated rows are identical in the relevant business columns.
#   We keep one row and log the issue as a warning.
#
# If distinct_versions > 1:
#   The same client_id appears with conflicting client information.
#   We reject those records as critical.

client_business_columns = [
    "client_name",
    "standardized_client_name",
    "industry",
    "region_id",
    "account_manager",
    "contract_type",
]

client_duplicate_versions_df = (
    clients_duplicate_rows_df
    .select(["client_id"] + client_business_columns)
    .distinct()
    .groupBy("client_id")
    .count()
    .withColumnRenamed("count", "distinct_versions")
)

client_exact_duplicate_ids_df = (
    client_duplicate_versions_df
    .filter(F.col("distinct_versions") == 1)
    .select("client_id")
)

client_conflicting_duplicate_ids_df = (
    client_duplicate_versions_df
    .filter(F.col("distinct_versions") > 1)
    .select("client_id")
)

# Log exact duplicated client_id records as warnings.
# One version will be kept in silver_clients_df.

clients_exact_duplicates_warning_df = (
    clients_duplicate_rows_df
    .join(client_exact_duplicate_ids_df, on="client_id", how="inner")
    .select(
        F.lit("bronze_clients").alias("source_table"),
        F.col("client_id").cast("string").alias("record_id"),
        F.lit("exact duplicated client_id").alias("rejection_reason"),
        F.lit("warning").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Log conflicting duplicated client_id records as critical.
# These records are excluded from silver_clients_df.

clients_conflicting_duplicates_rejected_df = (
    clients_duplicate_rows_df
    .join(client_conflicting_duplicate_ids_df, on="client_id", how="inner")
    .select(
        F.lit("bronze_clients").alias("source_table"),
        F.col("client_id").cast("string").alias("record_id"),
        F.lit("conflicting client_id records").alias("rejection_reason"),
        F.lit("critical").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Exclude conflicting duplicated client IDs before applying warning-level checks.

clients_valid_for_silver_df = (
    clients_valid_df
    .join(client_conflicting_duplicate_ids_df, on="client_id", how="left_anti")
)

# Warning condition: missing account manager.
# This is not a critical issue for workforce KPI calculation,
# so the record is kept in Silver but flagged for review.

clients_missing_manager_condition = (
    F.col("account_manager").isNull()
    | (F.col("account_manager") == "")
)

clients_missing_manager_warning_df = (
    clients_valid_for_silver_df
    .filter(clients_missing_manager_condition)
    .select(
        F.lit("bronze_clients").alias("source_table"),
        F.col("client_id").cast("string").alias("record_id"),
        F.lit("account_manager is missing").alias("rejection_reason"),
        F.lit("warning").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Detect potential duplicate client names using standardized_client_name.
# These records are not automatically merged because client deduplication
# requires business confirmation.

client_duplicate_names_df = (
    clients_valid_for_silver_df
    .groupBy("standardized_client_name")
    .count()
    .filter(F.col("count") > 1)
    .select("standardized_client_name")
)

clients_potential_duplicates_df = (
    clients_valid_for_silver_df
    .join(client_duplicate_names_df, on="standardized_client_name", how="inner")
)

clients_potential_duplicates_warning_df = (
    clients_potential_duplicates_df
    .select(
        F.lit("bronze_clients").alias("source_table"),
        F.col("client_id").cast("string").alias("record_id"),
        F.lit("potential duplicate client name").alias("rejection_reason"),
        F.lit("warning").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Combine all client issues:
# - critical records are excluded from Silver;
# - warning records are kept in Silver but flagged.

clients_issues_df = union_dataframes([
    clients_basic_critical_rejected_df,
    clients_exact_duplicates_warning_df,
    clients_conflicting_duplicates_rejected_df,
    clients_missing_manager_warning_df,
    clients_potential_duplicates_warning_df
])

# Create the Silver clients DataFrame.
# Critical records are excluded.
# Warning records are kept but flagged.
# Exact duplicated client_id records are deduplicated.

silver_clients_df = (
    clients_valid_for_silver_df
    .dropDuplicates(["client_id"])
    .withColumn(
        "account_manager_missing_flag",
        clients_missing_manager_condition
    )
    .join(
        client_duplicate_names_df.withColumn("potential_duplicate_flag", F.lit(True)),
        on="standardized_client_name",
        how="left"
    )
    .withColumn(
        "potential_duplicate_flag",
        F.coalesce(F.col("potential_duplicate_flag"), F.lit(False))
    )
    .select(
        "client_id",
        "client_name",
        "standardized_client_name",
        "industry",
        "region_id",
        "account_manager",
        "account_manager_missing_flag",
        "contract_type",
        "potential_duplicate_flag",
        "source_file",
        "ingestion_timestamp",
        "bronze_load_id"
    )
)

display(silver_clients_df)

client_id,client_name,standardized_client_name,industry,region_id,account_manager,account_manager_missing_flag,contract_type,potential_duplicate_flag,source_file,ingestion_timestamp,bronze_load_id
C001,Alpha Bank,alphabank,Banking,LATAM,Lucas Lopez,false,Managed Service,true,clients.xlsx,2026-05-09T12:44:30.507Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
C002,Nova Retail,novaretail,Retail,LATAM,Sofia Martinez,false,Managed Service,true,clients.xlsx,2026-05-09T12:44:30.507Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
C003,MediCore Health,medicorehealth,Banking,NA,Michael Thomas,false,Managed Service,false,clients.xlsx,2026-05-09T12:44:30.507Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
C004,TechNova Systems,technovasystems,Telecommunications,APAC,John Perez,false,Fixed Price,true,clients.xlsx,2026-05-09T12:44:30.507Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
C006,IronWorks Manufacturing,ironworksmanufacturing,Telecommunications,EMEA,Mia Sanchez,false,Fixed Price,false,clients.xlsx,2026-05-09T12:44:30.507Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
C007,GreenGrid Energy,greengridenergy,Manufacturing,EMEA,null,true,Managed Service,false,clients.xlsx,2026-05-09T12:44:30.507Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
C009,Andes Finance,andesfinance,Healthcare,EMEA,Mateo Martinez,false,Time and Materials,false,clients.xlsx,2026-05-09T12:44:30.507Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
C010,BrightMart,brightmart,Technology,NA,null,true,Fixed Price,false,clients.xlsx,2026-05-09T12:44:30.507Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
C011,CloudPath Technologies,cloudpathtechnologies,Banking,EMEA,Valentina Rodriguez,false,Fixed Price,false,clients.xlsx,2026-05-09T12:44:30.507Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
C012,PrimeCare Services,primecareservices,Manufacturing,NA,James Lopez,false,Fixed Price,false,clients.xlsx,2026-05-09T12:44:30.507Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


In [0]:
# Check the result of the client transformation.
# This quick validation compares the number of records from Bronze,
# the number of valid records kept in Silver, and the number of
# critical or warning records logged during the transformation.

bronze_clients_count = bronze_clients_df.count()
silver_clients_count = silver_clients_df.count()

print(f"Bronze clients rows: {bronze_clients_count}")
print(f"Silver clients rows: {silver_clients_count}")

if clients_issues_df is not None:
    client_issues_count = clients_issues_df.count()
    print(f"Client critical or warning records: {client_issues_count}")
    display(clients_issues_df)
else:
    print("Client critical or warning records: 0")

Bronze clients rows: 15
Silver clients rows: 13
Client critical or warning records: 10


source_table,record_id,rejection_reason,rejection_severity,rejected_at,bronze_load_id
bronze_clients,C005,client_region could not be normalized,critical,2026-05-09T12:46:25.871Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_clients,C008,client_region could not be normalized,critical,2026-05-09T12:46:25.871Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_clients,C007,account_manager is missing,warning,2026-05-09T12:46:25.871Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_clients,C010,account_manager is missing,warning,2026-05-09T12:46:25.871Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_clients,C001,potential duplicate client name,warning,2026-05-09T12:46:25.871Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_clients,C002,potential duplicate client name,warning,2026-05-09T12:46:25.871Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_clients,C004,potential duplicate client name,warning,2026-05-09T12:46:25.871Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_clients,C013,potential duplicate client name,warning,2026-05-09T12:46:25.871Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_clients,C014,potential duplicate client name,warning,2026-05-09T12:46:25.871Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_clients,C015,potential duplicate client name,warning,2026-05-09T12:46:25.871Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


%md
# Transform Assignments

## Source and target

Source table: `bronze_assignments`  
Target table: `silver_assignments`

## Transformation rules

- Trim identifier fields.
- Convert assignment dates to date type.
- Normalize assignment status values.
- Validate `assignment_id` as the primary identifier.
- Detect duplicated `assignment_id` values.
- Validate `employee_id` against `silver_employees`.
- Validate `client_id` against `silver_clients`.
- Validate `allocation_percentage`.
- Validate assignment date ranges.

## Critical rejection rules

Records are registered as critical and excluded from `silver_assignments` when:

- `assignment_id` is missing.
- The same `assignment_id` appears with conflicting assignment information.
- `employee_id` is missing.
- `employee_id` does not exist in `silver_employees`.
- `client_id` is missing.
- `client_id` does not exist in `silver_clients`.
- `allocation_percentage` is less than or equal to 0 or greater than 100.
- `assignment_start_date` is missing.
- `assignment_end_date` is earlier than `assignment_start_date`.
- `assignment_status` cannot be normalized.

## Warning rules

Records are registered as warnings but deduplicated when:

- The same `assignment_id` appears more than once with identical business information.

## Notes

Assignments depend on valid employees and clients. For that reason, `silver_assignments` is built after `silver_employees` and `silver_clients`.

In [0]:
# Transform assignment data from Bronze to Silver.
# This section validates assignment identifiers, foreign keys, allocation values,
# date ranges, duplicated assignment IDs, and creates silver_assignments_df.

assignments_base_df = (
    bronze_assignments_df
    .withColumn("assignment_id", F.trim(F.col("assignment_id")))
    .withColumn("employee_id", F.trim(F.col("employee_id")))
    .withColumn("client_id", F.trim(F.col("client_id")))
    .withColumn("project_id", F.trim(F.col("project_id")))
    .withColumn("assignment_start_date", F.to_date(F.col("assignment_start_date")))
    .withColumn("assignment_end_date", F.to_date(F.col("assignment_end_date")))
    .withColumn("allocation_percentage", F.col("allocation_percentage").cast("double"))
    .withColumn("assignment_status", normalize_assignment_status(F.col("assignment_status")))
)

# Reference valid employee and client IDs from already cleaned Silver DataFrames.

valid_employee_ids_df = silver_employees_df.select("employee_id").distinct()
valid_client_ids_df = silver_clients_df.select("client_id").distinct()

# Critical rejection conditions.

assignments_invalid_id_condition = (
    F.col("assignment_id").isNull()
    | (F.col("assignment_id") == "")
)

assignments_invalid_employee_id_condition = (
    F.col("employee_id").isNull()
    | (F.col("employee_id") == "")
)

assignments_invalid_client_id_condition = (
    F.col("client_id").isNull()
    | (F.col("client_id") == "")
)

assignments_invalid_allocation_condition = (
    F.col("allocation_percentage").isNull()
    | (F.col("allocation_percentage") <= 0)
    | (F.col("allocation_percentage") > 100)
)

assignments_missing_start_date_condition = (
    F.col("assignment_start_date").isNull()
)

assignments_invalid_date_range_condition = (
    F.col("assignment_end_date").isNotNull()
    & F.col("assignment_start_date").isNotNull()
    & (F.col("assignment_end_date") < F.col("assignment_start_date"))
)

assignments_invalid_status_condition = (
    F.col("assignment_status").isNull()
)

# Create rejected records for basic critical assignment issues.

assignments_basic_critical_rejections = [
    create_rejected_records(
        assignments_base_df,
        source_table="bronze_assignments",
        record_id_column="assignment_id",
        condition=assignments_invalid_id_condition,
        reason="assignment_id is missing"
    ),
    create_rejected_records(
        assignments_base_df,
        source_table="bronze_assignments",
        record_id_column="assignment_id",
        condition=assignments_invalid_employee_id_condition,
        reason="employee_id is missing"
    ),
    create_rejected_records(
        assignments_base_df,
        source_table="bronze_assignments",
        record_id_column="assignment_id",
        condition=assignments_invalid_client_id_condition,
        reason="client_id is missing"
    ),
    create_rejected_records(
        assignments_base_df,
        source_table="bronze_assignments",
        record_id_column="assignment_id",
        condition=assignments_invalid_allocation_condition,
        reason="allocation_percentage is invalid"
    ),
    create_rejected_records(
        assignments_base_df,
        source_table="bronze_assignments",
        record_id_column="assignment_id",
        condition=assignments_missing_start_date_condition,
        reason="assignment_start_date is missing"
    ),
    create_rejected_records(
        assignments_base_df,
        source_table="bronze_assignments",
        record_id_column="assignment_id",
        condition=assignments_invalid_date_range_condition,
        reason="assignment_end_date is earlier than assignment_start_date"
    ),
    create_rejected_records(
        assignments_base_df,
        source_table="bronze_assignments",
        record_id_column="assignment_id",
        condition=assignments_invalid_status_condition,
        reason="assignment_status is invalid"
    ),
]

assignments_basic_critical_rejected_df = union_dataframes(assignments_basic_critical_rejections)

# Keep only records that do not have basic critical issues.

assignments_valid_base_df = (
    assignments_base_df
    .filter(~assignments_invalid_id_condition)
    .filter(~assignments_invalid_employee_id_condition)
    .filter(~assignments_invalid_client_id_condition)
    .filter(~assignments_invalid_allocation_condition)
    .filter(~assignments_missing_start_date_condition)
    .filter(~assignments_invalid_date_range_condition)
    .filter(~assignments_invalid_status_condition)
)

# Validate employee_id foreign key against silver_employees.

assignments_invalid_employee_fk_df = (
    assignments_valid_base_df
    .join(valid_employee_ids_df, on="employee_id", how="left_anti")
)

assignments_invalid_employee_fk_rejected_df = (
    assignments_invalid_employee_fk_df
    .select(
        F.lit("bronze_assignments").alias("source_table"),
        F.col("assignment_id").cast("string").alias("record_id"),
        F.lit("employee_id does not exist in silver_employees").alias("rejection_reason"),
        F.lit("critical").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Validate client_id foreign key against silver_clients.

assignments_invalid_client_fk_df = (
    assignments_valid_base_df
    .join(valid_client_ids_df, on="client_id", how="left_anti")
)

assignments_invalid_client_fk_rejected_df = (
    assignments_invalid_client_fk_df
    .select(
        F.lit("bronze_assignments").alias("source_table"),
        F.col("assignment_id").cast("string").alias("record_id"),
        F.lit("client_id does not exist in silver_clients").alias("rejection_reason"),
        F.lit("critical").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Keep only records with valid foreign keys.

assignments_valid_fk_df = (
    assignments_valid_base_df
    .join(valid_employee_ids_df, on="employee_id", how="inner")
    .join(valid_client_ids_df, on="client_id", how="inner")
)

# Detect duplicated assignment_id values.

assignment_duplicate_ids_df = (
    assignments_valid_fk_df
    .groupBy("assignment_id")
    .count()
    .filter(F.col("count") > 1)
    .select("assignment_id")
)

assignments_duplicate_rows_df = (
    assignments_valid_fk_df
    .join(assignment_duplicate_ids_df, on="assignment_id", how="inner")
)

assignment_business_columns = [
    "employee_id",
    "client_id",
    "project_id",
    "assignment_start_date",
    "assignment_end_date",
    "allocation_percentage",
    "assignment_status",
]

assignment_duplicate_versions_df = (
    assignments_duplicate_rows_df
    .select(["assignment_id"] + assignment_business_columns)
    .distinct()
    .groupBy("assignment_id")
    .count()
    .withColumnRenamed("count", "distinct_versions")
)

assignment_exact_duplicate_ids_df = (
    assignment_duplicate_versions_df
    .filter(F.col("distinct_versions") == 1)
    .select("assignment_id")
)

assignment_conflicting_duplicate_ids_df = (
    assignment_duplicate_versions_df
    .filter(F.col("distinct_versions") > 1)
    .select("assignment_id")
)

assignments_exact_duplicates_warning_df = (
    assignments_duplicate_rows_df
    .join(assignment_exact_duplicate_ids_df, on="assignment_id", how="inner")
    .select(
        F.lit("bronze_assignments").alias("source_table"),
        F.col("assignment_id").cast("string").alias("record_id"),
        F.lit("exact duplicated assignment_id").alias("rejection_reason"),
        F.lit("warning").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

assignments_conflicting_duplicates_rejected_df = (
    assignments_duplicate_rows_df
    .join(assignment_conflicting_duplicate_ids_df, on="assignment_id", how="inner")
    .select(
        F.lit("bronze_assignments").alias("source_table"),
        F.col("assignment_id").cast("string").alias("record_id"),
        F.lit("conflicting assignment_id records").alias("rejection_reason"),
        F.lit("critical").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Combine all assignment issues.

assignments_rejected_df = union_dataframes([
    assignments_basic_critical_rejected_df,
    assignments_invalid_employee_fk_rejected_df,
    assignments_invalid_client_fk_rejected_df,
    assignments_exact_duplicates_warning_df,
    assignments_conflicting_duplicates_rejected_df
])

# Exclude conflicting duplicated assignment IDs from Silver.
# Exact duplicates are deduplicated.

assignments_valid_for_silver_df = (
    assignments_valid_fk_df
    .join(assignment_conflicting_duplicate_ids_df, on="assignment_id", how="left_anti")
)

silver_assignments_df = (
    assignments_valid_for_silver_df
    .dropDuplicates(["assignment_id"])
    .select(
        "assignment_id",
        "employee_id",
        "client_id",
        "project_id",
        "assignment_start_date",
        "assignment_end_date",
        "allocation_percentage",
        "assignment_status",
        "source_file",
        "ingestion_timestamp",
        "bronze_load_id"
    )
)

display(silver_assignments_df.limit(10))

assignment_id,employee_id,client_id,project_id,assignment_start_date,assignment_end_date,allocation_percentage,assignment_status,source_file,ingestion_timestamp,bronze_load_id
A00003,E0002,C001,P011,2026-05-16,null,75.0,active,assignments.xlsx,2026-05-09T12:44:15.640Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
A00004,E0003,C012,P035,2026-07-24,null,50.0,active,assignments.xlsx,2026-05-09T12:44:15.640Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
A00007,E0006,C001,P007,2026-06-11,null,100.0,active,assignments.xlsx,2026-05-09T12:44:15.640Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
A00012,E0011,C002,P029,2026-01-07,null,75.0,active,assignments.xlsx,2026-05-09T12:44:15.640Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
A00033,E0031,C001,P003,2026-06-07,null,75.0,active,assignments.xlsx,2026-05-09T12:44:15.640Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
A00044,E0040,C001,P009,2026-03-26,null,50.0,active,assignments.xlsx,2026-05-09T12:44:15.640Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
A00111,E0102,C007,P029,2026-02-14,2026-10-29,75.0,ended,assignments.xlsx,2026-05-09T12:44:15.640Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
A00112,E0103,C012,P027,2026-03-25,null,50.0,active,assignments.xlsx,2026-05-09T12:44:15.640Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
A00175,E0159,C010,P033,2026-06-16,2026-12-31,75.0,ended,assignments.xlsx,2026-05-09T12:44:15.640Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
A00176,E0160,C001,P021,2026-05-17,2026-10-14,75.0,ended,assignments.xlsx,2026-05-09T12:44:15.640Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


In [0]:
# Check the result of the assignment transformation.
# This quick validation compares the number of records from Bronze,
# the number of valid records kept in Silver, and the number of
# critical or warning records logged during the transformation.

bronze_assignments_count = bronze_assignments_df.count()
silver_assignments_count = silver_assignments_df.count()

print(f"Bronze assignments rows: {bronze_assignments_count}")
print(f"Silver assignments rows: {silver_assignments_count}")

if assignments_rejected_df is not None:
    rejected_assignments_count = assignments_rejected_df.count()
    print(f"Assignment critical or warning records: {rejected_assignments_count}")
    display(assignments_rejected_df)
else:
    print("Assignment critical or warning records: 0")

Bronze assignments rows: 555
Silver assignments rows: 435
Assignment critical or warning records: 123


source_table,record_id,rejection_reason,rejection_severity,rejected_at,bronze_load_id
bronze_assignments,A00036,allocation_percentage is invalid,critical,2026-05-09T12:47:03.197Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_assignments,A00150,allocation_percentage is invalid,critical,2026-05-09T12:47:03.197Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_assignments,A00317,allocation_percentage is invalid,critical,2026-05-09T12:47:03.197Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_assignments,A00328,allocation_percentage is invalid,critical,2026-05-09T12:47:03.197Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_assignments,A00515,allocation_percentage is invalid,critical,2026-05-09T12:47:03.197Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_assignments,A00127,assignment_end_date is earlier than assignment_start_date,critical,2026-05-09T12:47:03.197Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_assignments,A00281,assignment_end_date is earlier than assignment_start_date,critical,2026-05-09T12:47:03.197Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_assignments,A00490,assignment_end_date is earlier than assignment_start_date,critical,2026-05-09T12:47:03.197Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_assignments,A00019,employee_id does not exist in silver_employees,critical,2026-05-09T12:47:03.197Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_assignments,A00099,employee_id does not exist in silver_employees,critical,2026-05-09T12:47:03.197Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


In [0]:
# Summarize assignment issues by severity and rejection reason.
# This helps understand why assignment records were excluded from Silver.

assignment_issues_summary_df = (
    assignments_rejected_df
    .groupBy("rejection_severity", "rejection_reason")
    .agg(F.count("*").alias("records_count"))
    .orderBy("rejection_severity", "rejection_reason")
)

display(assignment_issues_summary_df)

rejection_severity,rejection_reason,records_count
critical,allocation_percentage is invalid,5
critical,assignment_end_date is earlier than assignment_start_date,3
critical,client_id does not exist in silver_clients,103
critical,employee_id does not exist in silver_employees,12


%md
# Transform Hours

## Source and target

Source table: `bronze_hours`  
Target table: `silver_hours`

## Transformation rules

- Trim identifier fields.
- Normalize `period` into `YYYY-MM` format.
- Convert hour columns to numeric type.
- Validate `employee_id` against `silver_employees`.
- Validate `client_id` against `silver_clients`.
- Validate worked, available, billable, non-billable and overtime hours.

## Critical rejection rules

Records are registered as critical and excluded from `silver_hours` when:

- `period` cannot be normalized.
- `employee_id` is missing.
- `employee_id` does not exist in `silver_employees`.
- `client_id` is missing.
- `client_id` does not exist in `silver_clients`.
- `worked_hours` is negative.
- `available_hours` is missing or less than or equal to 0.
- `billable_hours` is greater than `worked_hours`.
- `non_billable_hours` is negative.
- `overtime_hours` is negative.

## Notes

Hours data depends on valid employees and clients. Records referencing invalid master data are excluded to preserve referential integrity in the analytical model.

In [0]:
# Transform hours data from Bronze to Silver.
# This section normalizes period values, creates a logical record identifier,
# validates hour metrics, validates employee/client foreign keys,
# detects duplicated logical hour records, and creates silver_hours_df.

hours_base_df = (
    bronze_hours_df
    .withColumn("period", F.trim(F.col("period")))
    .withColumn("employee_id", F.trim(F.col("employee_id")))
    .withColumn("client_id", F.trim(F.col("client_id")))
    .withColumn("project_id", F.trim(F.col("project_id")))
    .withColumn("available_hours", F.col("available_hours").cast("double"))
    .withColumn("worked_hours", F.col("worked_hours").cast("double"))
    .withColumn("billable_hours", F.col("billable_hours").cast("double"))
    .withColumn("non_billable_hours", F.col("non_billable_hours").cast("double"))
    .withColumn("overtime_hours", F.col("overtime_hours").cast("double"))
)

# Normalize period values into YYYY-MM.
# Accepted examples:
# - 2026-01
# - 2026/01
# - 202601
# - 2026.04
#
# Values like Jan-2026 are considered invalid for this pipeline version.

hours_base_df = (
    hours_base_df
    .withColumn(
        "period_normalized",
        F.when(F.col("period").rlike(r"^\d{4}-\d{2}$"), F.col("period"))
         .when(F.col("period").rlike(r"^\d{4}/\d{2}$"), F.regexp_replace(F.col("period"), "/", "-"))
         .when(F.col("period").rlike(r"^\d{6}$"), F.concat(F.substring(F.col("period"), 1, 4), F.lit("-"), F.substring(F.col("period"), 5, 2)))
         .when(F.col("period").rlike(r"^\d{4}\.\d{2}$"), F.regexp_replace(F.col("period"), r"\.", "-"))
         .otherwise(None)
    )
)

# Create a logical record identifier for hour records.
# Hours do not have a single source primary key, so we use:
# period + employee_id + client_id + project_id.
#
# We use coalesce to keep a traceable record_id even when some fields are invalid or missing.

hours_base_df = (
    hours_base_df
    .withColumn(
        "hours_record_id",
        F.concat_ws(
            "|",
            F.coalesce(F.col("period_normalized"), F.col("period"), F.lit("UNKNOWN_PERIOD")),
            F.coalesce(F.col("employee_id"), F.lit("UNKNOWN_EMPLOYEE")),
            F.coalesce(F.col("client_id"), F.lit("UNKNOWN_CLIENT")),
            F.coalesce(F.col("project_id"), F.lit("UNKNOWN_PROJECT"))
        )
    )
)

# Reference valid employee and client IDs from cleaned Silver DataFrames.

valid_employee_ids_df = silver_employees_df.select("employee_id").distinct()
valid_client_ids_df = silver_clients_df.select("client_id").distinct()

# Critical rejection conditions.

hours_invalid_period_condition = F.col("period_normalized").isNull()

hours_invalid_employee_id_condition = (
    F.col("employee_id").isNull()
    | (F.col("employee_id") == "")
)

hours_invalid_client_id_condition = (
    F.col("client_id").isNull()
    | (F.col("client_id") == "")
)

hours_negative_worked_condition = (
    F.col("worked_hours").isNull()
    | (F.col("worked_hours") < 0)
)

hours_invalid_available_condition = (
    F.col("available_hours").isNull()
    | (F.col("available_hours") <= 0)
)

hours_invalid_billable_condition = (
    F.col("billable_hours").isNull()
    | (F.col("billable_hours") < 0)
    | (F.col("billable_hours") > F.col("worked_hours"))
)

hours_invalid_non_billable_condition = (
    F.col("non_billable_hours").isNull()
    | (F.col("non_billable_hours") < 0)
)

hours_invalid_overtime_condition = (
    F.col("overtime_hours").isNull()
    | (F.col("overtime_hours") < 0)
)

# Create rejected records for basic critical hour issues.
# All hour issues use hours_record_id as record_id for better traceability.

hours_basic_critical_rejections = [
    create_rejected_records(
        hours_base_df,
        source_table="bronze_hours",
        record_id_column="hours_record_id",
        condition=hours_invalid_period_condition,
        reason="period could not be normalized"
    ),
    create_rejected_records(
        hours_base_df,
        source_table="bronze_hours",
        record_id_column="hours_record_id",
        condition=hours_invalid_employee_id_condition,
        reason="employee_id is missing"
    ),
    create_rejected_records(
        hours_base_df,
        source_table="bronze_hours",
        record_id_column="hours_record_id",
        condition=hours_invalid_client_id_condition,
        reason="client_id is missing"
    ),
    create_rejected_records(
        hours_base_df,
        source_table="bronze_hours",
        record_id_column="hours_record_id",
        condition=hours_negative_worked_condition,
        reason="worked_hours is negative or missing"
    ),
    create_rejected_records(
        hours_base_df,
        source_table="bronze_hours",
        record_id_column="hours_record_id",
        condition=hours_invalid_available_condition,
        reason="available_hours is missing or invalid"
    ),
    create_rejected_records(
        hours_base_df,
        source_table="bronze_hours",
        record_id_column="hours_record_id",
        condition=hours_invalid_billable_condition,
        reason="billable_hours is invalid or greater than worked_hours"
    ),
    create_rejected_records(
        hours_base_df,
        source_table="bronze_hours",
        record_id_column="hours_record_id",
        condition=hours_invalid_non_billable_condition,
        reason="non_billable_hours is invalid"
    ),
    create_rejected_records(
        hours_base_df,
        source_table="bronze_hours",
        record_id_column="hours_record_id",
        condition=hours_invalid_overtime_condition,
        reason="overtime_hours is invalid"
    ),
]

hours_basic_critical_rejected_df = union_dataframes(hours_basic_critical_rejections)

# Keep only records that do not have basic critical issues.

hours_valid_base_df = (
    hours_base_df
    .filter(~hours_invalid_period_condition)
    .filter(~hours_invalid_employee_id_condition)
    .filter(~hours_invalid_client_id_condition)
    .filter(~hours_negative_worked_condition)
    .filter(~hours_invalid_available_condition)
    .filter(~hours_invalid_billable_condition)
    .filter(~hours_invalid_non_billable_condition)
    .filter(~hours_invalid_overtime_condition)
)

# Validate employee_id foreign key against silver_employees.

hours_invalid_employee_fk_df = (
    hours_valid_base_df
    .join(valid_employee_ids_df, on="employee_id", how="left_anti")
)

hours_invalid_employee_fk_rejected_df = (
    hours_invalid_employee_fk_df
    .select(
        F.lit("bronze_hours").alias("source_table"),
        F.col("hours_record_id").cast("string").alias("record_id"),
        F.lit("employee_id does not exist in silver_employees").alias("rejection_reason"),
        F.lit("critical").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Validate client_id foreign key against silver_clients.

hours_invalid_client_fk_df = (
    hours_valid_base_df
    .join(valid_client_ids_df, on="client_id", how="left_anti")
)

hours_invalid_client_fk_rejected_df = (
    hours_invalid_client_fk_df
    .select(
        F.lit("bronze_hours").alias("source_table"),
        F.col("hours_record_id").cast("string").alias("record_id"),
        F.lit("client_id does not exist in silver_clients").alias("rejection_reason"),
        F.lit("critical").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Combine basic hour issues and foreign key issues.

hours_rejected_df = union_dataframes([
    hours_basic_critical_rejected_df,
    hours_invalid_employee_fk_rejected_df,
    hours_invalid_client_fk_rejected_df
])

# Keep only records with valid foreign keys.

hours_valid_for_silver_df = (
    hours_valid_base_df
    .join(valid_employee_ids_df, on="employee_id", how="inner")
    .join(valid_client_ids_df, on="client_id", how="inner")
)

# Detect duplicated logical hour records using hours_record_id.

hour_duplicate_ids_df = (
    hours_valid_for_silver_df
    .groupBy("hours_record_id")
    .count()
    .filter(F.col("count") > 1)
    .select("hours_record_id")
)

hours_duplicate_rows_df = (
    hours_valid_for_silver_df
    .join(hour_duplicate_ids_df, on="hours_record_id", how="inner")
)

# Distinguish exact duplicated hour records from conflicting duplicated hour records.
#
# If distinct_versions = 1:
#   The duplicated rows are identical in the relevant business columns.
#   We keep one row and log the issue as a warning.
#
# If distinct_versions > 1:
#   The same logical hour record appears with conflicting hour information.
#   We reject those records as critical.

hour_business_columns = [
    "period_normalized",
    "employee_id",
    "client_id",
    "project_id",
    "available_hours",
    "worked_hours",
    "billable_hours",
    "non_billable_hours",
    "overtime_hours",
]

hour_duplicate_versions_df = (
    hours_duplicate_rows_df
    .select(["hours_record_id"] + hour_business_columns)
    .distinct()
    .groupBy("hours_record_id")
    .count()
    .withColumnRenamed("count", "distinct_versions")
)

hour_exact_duplicate_ids_df = (
    hour_duplicate_versions_df
    .filter(F.col("distinct_versions") == 1)
    .select("hours_record_id")
)

hour_conflicting_duplicate_ids_df = (
    hour_duplicate_versions_df
    .filter(F.col("distinct_versions") > 1)
    .select("hours_record_id")
)

hours_exact_duplicates_warning_df = (
    hours_duplicate_rows_df
    .join(hour_exact_duplicate_ids_df, on="hours_record_id", how="inner")
    .select(
        F.lit("bronze_hours").alias("source_table"),
        F.col("hours_record_id").cast("string").alias("record_id"),
        F.lit("exact duplicated hour record").alias("rejection_reason"),
        F.lit("warning").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

hours_conflicting_duplicates_rejected_df = (
    hours_duplicate_rows_df
    .join(hour_conflicting_duplicate_ids_df, on="hours_record_id", how="inner")
    .select(
        F.lit("bronze_hours").alias("source_table"),
        F.col("hours_record_id").cast("string").alias("record_id"),
        F.lit("conflicting hour records for same period, employee, client and project").alias("rejection_reason"),
        F.lit("critical").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Add hour duplicate issues to the existing hours rejected records.

hours_rejected_df = union_dataframes([
    hours_rejected_df,
    hours_exact_duplicates_warning_df,
    hours_conflicting_duplicates_rejected_df
])

# Exclude conflicting duplicated logical keys from Silver.
# Exact duplicates are deduplicated.

hours_valid_for_silver_df = (
    hours_valid_for_silver_df
    .join(hour_conflicting_duplicate_ids_df, on="hours_record_id", how="left_anti")
)

# Create the Silver hours DataFrame.

silver_hours_df = (
    hours_valid_for_silver_df
    .dropDuplicates(["hours_record_id"])
    .select(
        "hours_record_id",
        F.col("period_normalized").alias("period"),
        "employee_id",
        "client_id",
        "project_id",
        "available_hours",
        "worked_hours",
        "billable_hours",
        "non_billable_hours",
        "overtime_hours",
        "source_file",
        "ingestion_timestamp",
        "bronze_load_id"
    )
)

display(silver_hours_df.limit(10))

hours_record_id,period,employee_id,client_id,project_id,available_hours,worked_hours,billable_hours,non_billable_hours,overtime_hours,source_file,ingestion_timestamp,bronze_load_id
2026-10|E0003|C012|P035,2026-10,E0003,C012,P035,160.0,144.0,111.0,33.0,0.0,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-10|E0124|C012|P008,2026-10,E0124,C012,P008,160.0,112.0,93.0,19.0,0.0,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-10|E0111|C006|P033,2026-10,E0111,C006,P033,160.0,158.0,116.0,42.0,0.0,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-11|E0328|C004|P033,2026-11,E0328,C004,P033,160.0,169.0,113.0,56.0,9.0,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-11|E0077|C012|P005,2026-11,E0077,C012,P005,160.0,99.0,73.0,26.0,0.0,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-11|E0315|C002|P007,2026-11,E0315,C002,P007,160.0,137.0,113.0,24.0,0.0,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-11|E0422|C002|P033,2026-11,E0422,C002,P033,160.0,143.0,134.0,9.0,0.0,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-11|E0220|C006|P032,2026-11,E0220,C006,P032,160.0,127.0,114.0,13.0,0.0,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-11|E0356|C001|P022,2026-11,E0356,C001,P022,160.0,135.0,103.0,32.0,0.0,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-11|E0192|C007|P003,2026-11,E0192,C007,P003,160.0,130.0,89.0,41.0,0.0,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


In [0]:
# Check the result of the hours transformation.
# This quick validation compares the number of records from Bronze,
# the number of valid records kept in Silver, and the number of
# critical records logged during the transformation.

bronze_hours_count = bronze_hours_df.count()
silver_hours_count = silver_hours_df.count()

print(f"Bronze hours rows: {bronze_hours_count}")
print(f"Silver hours rows: {silver_hours_count}")

if hours_rejected_df is not None:
    rejected_hours_count = hours_rejected_df.count()
    print(f"Hour critical records: {rejected_hours_count}")
    display(hours_rejected_df)
else:
    print("Hour critical records: 0")

Bronze hours rows: 4154
Silver hours rows: 3292
Hour critical records: 873


source_table,record_id,rejection_reason,rejection_severity,rejected_at,bronze_load_id
bronze_hours,Jan-2026|E0250|C008|P028,period could not be normalized,critical,2026-05-09T12:48:05.496Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_hours,2026-11|E0299|C012|P011,worked_hours is negative or missing,critical,2026-05-09T12:48:05.496Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_hours,2026-03|E0029|C012|P013,worked_hours is negative or missing,critical,2026-05-09T12:48:05.496Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_hours,2026-04|E0153|C009|P004,worked_hours is negative or missing,critical,2026-05-09T12:48:05.496Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_hours,2026-04|E0170|C011|P012,worked_hours is negative or missing,critical,2026-05-09T12:48:05.496Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_hours,2026-04|E0126|C010|P004,worked_hours is negative or missing,critical,2026-05-09T12:48:05.496Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_hours,2026-09|E0018|C010|P026,available_hours is missing or invalid,critical,2026-05-09T12:48:05.496Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_hours,2026-10|E0116|C005|P018,available_hours is missing or invalid,critical,2026-05-09T12:48:05.496Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_hours,2026-05|E0072|C002|P014,available_hours is missing or invalid,critical,2026-05-09T12:48:05.496Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_hours,2026-11|E0299|C012|P011,billable_hours is invalid or greater than worked_hours,critical,2026-05-09T12:48:05.496Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


In [0]:
# Summarize hour issues by severity and rejection reason.
# This helps understand why hour records were excluded from Silver.

hour_issues_summary_df = (
    hours_rejected_df
    .groupBy("rejection_severity", "rejection_reason")
    .agg(F.count("*").alias("records_count"))
    .orderBy("rejection_severity", "rejection_reason")
)

display(hour_issues_summary_df)

rejection_severity,rejection_reason,records_count
critical,available_hours is missing or invalid,3
critical,billable_hours is invalid or greater than worked_hours,10
critical,client_id does not exist in silver_clients,801
critical,"conflicting hour records for same period, employee, client and project",2
critical,employee_id does not exist in silver_employees,46
critical,non_billable_hours is invalid,5
critical,period could not be normalized,1
critical,worked_hours is negative or missing,5


%md
# Transform Costs

## Source and target

Source table: `bronze_costs`  
Target table: `silver_costs`

## Transformation rules

- Trim identifier fields.
- Normalize `period` into `YYYY-MM` format.
- Normalize region values into `region_id`.
- Normalize currency values.
- Convert cost fields to numeric type.
- Validate `employee_id` against `silver_employees`.
- Validate cost values.
- Validate that `total_cost` is consistent with `salary_cost + benefits_cost`.
- Detect duplicated logical cost records using the composite key `period + employee_id`.

## Critical rejection rules

Records are registered as critical and excluded from `silver_costs` when:

- `period` cannot be normalized.
- `employee_id` is missing.
- `employee_id` does not exist in `silver_employees`.
- `region_id` cannot be normalized.
- `currency` cannot be normalized.
- `salary_cost` is missing or negative.
- `benefits_cost` is missing or negative.
- `total_cost` is missing or negative.
- `total_cost` is not consistent with `salary_cost + benefits_cost`.
- The same `period + employee_id` appears with conflicting cost information.

## Warning rules

Records are registered as warnings but deduplicated when:

- The same `period + employee_id` appears more than once with identical cost information.

## Notes

Costs do not have a single source primary key. For this reason, a logical composite key is created using `period + employee_id`.

In [0]:

costs_base_df = (
    bronze_costs_df
    .withColumn("period", F.trim(F.col("period")))
    .withColumn("employee_id", F.trim(F.col("employee_id")))
    .withColumn("region_id", normalize_region(F.col("region")))
    .withColumn("salary_cost", F.col("salary_cost").cast("double"))
    .withColumn("benefits_cost", F.col("benefits_cost").cast("double"))
    .withColumn("total_cost", F.col("total_cost").cast("double"))
    .withColumn("currency", normalize_currency(F.col("currency")))
)

# Normalize period values into YYYY-MM.
# Accepted examples:
# - 2026-01
# - 2026/01
# - 202601
# - 2026.04

costs_base_df = (
    costs_base_df
    .withColumn(
        "period_normalized",
        F.when(F.col("period").rlike(r"^\d{4}-\d{2}$"), F.col("period"))
         .when(F.col("period").rlike(r"^\d{4}/\d{2}$"), F.regexp_replace(F.col("period"), "/", "-"))
         .when(F.col("period").rlike(r"^\d{6}$"), F.concat(F.substring(F.col("period"), 1, 4), F.lit("-"), F.substring(F.col("period"), 5, 2)))
         .when(F.col("period").rlike(r"^\d{4}\.\d{2}$"), F.regexp_replace(F.col("period"), r"\.", "-"))
         .otherwise(None)
    )
)

# Create a logical record identifier for cost records.
# Costs do not have a single source primary key, so we use:
# period + employee_id

costs_base_df = (
    costs_base_df
    .withColumn(
        "cost_record_id",
        F.concat_ws("|", F.col("period_normalized"), F.col("employee_id"))
    )
)

# Reference valid employees from the cleaned Silver employees DataFrame.

valid_employee_ids_df = silver_employees_df.select("employee_id").distinct()

# Critical rejection conditions.

costs_invalid_period_condition = F.col("period_normalized").isNull()

costs_invalid_employee_id_condition = (
    F.col("employee_id").isNull()
    | (F.col("employee_id") == "")
)

costs_invalid_region_condition = F.col("region_id").isNull()

costs_invalid_currency_condition = F.col("currency").isNull()

costs_invalid_salary_condition = (
    F.col("salary_cost").isNull()
    | (F.col("salary_cost") < 0)
)

costs_invalid_benefits_condition = (
    F.col("benefits_cost").isNull()
    | (F.col("benefits_cost") < 0)
)

costs_invalid_total_condition = (
    F.col("total_cost").isNull()
    | (F.col("total_cost") < 0)
)

# A small tolerance is used because cost values are decimals.
# The total cost should approximately match salary_cost + benefits_cost.

costs_inconsistent_total_condition = (
    F.col("salary_cost").isNotNull()
    & F.col("benefits_cost").isNotNull()
    & F.col("total_cost").isNotNull()
    & (F.abs(F.col("total_cost") - (F.col("salary_cost") + F.col("benefits_cost"))) > 0.01)
)

# Create rejected records for basic critical cost issues.

costs_basic_critical_rejections = [
    create_rejected_records(
        costs_base_df,
        source_table="bronze_costs",
        record_id_column="cost_record_id",
        condition=costs_invalid_period_condition,
        reason="period could not be normalized"
    ),
    create_rejected_records(
        costs_base_df,
        source_table="bronze_costs",
        record_id_column="cost_record_id",
        condition=costs_invalid_employee_id_condition,
        reason="employee_id is missing"
    ),
    create_rejected_records(
        costs_base_df,
        source_table="bronze_costs",
        record_id_column="cost_record_id",
        condition=costs_invalid_region_condition,
        reason="region could not be normalized"
    ),
    create_rejected_records(
        costs_base_df,
        source_table="bronze_costs",
        record_id_column="cost_record_id",
        condition=costs_invalid_currency_condition,
        reason="currency could not be normalized"
    ),
    create_rejected_records(
        costs_base_df,
        source_table="bronze_costs",
        record_id_column="cost_record_id",
        condition=costs_invalid_salary_condition,
        reason="salary_cost is missing or negative"
    ),
    create_rejected_records(
        costs_base_df,
        source_table="bronze_costs",
        record_id_column="cost_record_id",
        condition=costs_invalid_benefits_condition,
        reason="benefits_cost is missing or negative"
    ),
    create_rejected_records(
        costs_base_df,
        source_table="bronze_costs",
        record_id_column="cost_record_id",
        condition=costs_invalid_total_condition,
        reason="total_cost is missing or negative"
    ),
    create_rejected_records(
        costs_base_df,
        source_table="bronze_costs",
        record_id_column="cost_record_id",
        condition=costs_inconsistent_total_condition,
        reason="total_cost is inconsistent with salary_cost plus benefits_cost"
    ),
]

costs_basic_critical_rejected_df = union_dataframes(costs_basic_critical_rejections)

# Keep only records that do not have basic critical issues.

costs_valid_base_df = (
    costs_base_df
    .filter(~costs_invalid_period_condition)
    .filter(~costs_invalid_employee_id_condition)
    .filter(~costs_invalid_region_condition)
    .filter(~costs_invalid_currency_condition)
    .filter(~costs_invalid_salary_condition)
    .filter(~costs_invalid_benefits_condition)
    .filter(~costs_invalid_total_condition)
    .filter(~costs_inconsistent_total_condition)
)

# Validate employee_id foreign key against silver_employees.

costs_invalid_employee_fk_df = (
    costs_valid_base_df
    .join(valid_employee_ids_df, on="employee_id", how="left_anti")
)

costs_invalid_employee_fk_rejected_df = (
    costs_invalid_employee_fk_df
    .select(
        F.lit("bronze_costs").alias("source_table"),
        F.col("cost_record_id").cast("string").alias("record_id"),
        F.lit("employee_id does not exist in silver_employees").alias("rejection_reason"),
        F.lit("critical").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Keep only records with valid employee foreign keys.

costs_valid_fk_df = (
    costs_valid_base_df
    .join(valid_employee_ids_df, on="employee_id", how="inner")
)

# Detect duplicated logical cost records using cost_record_id.

cost_duplicate_ids_df = (
    costs_valid_fk_df
    .groupBy("cost_record_id")
    .count()
    .filter(F.col("count") > 1)
    .select("cost_record_id")
)

costs_duplicate_rows_df = (
    costs_valid_fk_df
    .join(cost_duplicate_ids_df, on="cost_record_id", how="inner")
)

# Distinguish exact duplicated cost records from conflicting duplicated cost records.

cost_business_columns = [
    "period_normalized",
    "employee_id",
    "region_id",
    "salary_cost",
    "benefits_cost",
    "total_cost",
    "currency",
]

cost_duplicate_versions_df = (
    costs_duplicate_rows_df
    .select(["cost_record_id"] + cost_business_columns)
    .distinct()
    .groupBy("cost_record_id")
    .count()
    .withColumnRenamed("count", "distinct_versions")
)

cost_exact_duplicate_ids_df = (
    cost_duplicate_versions_df
    .filter(F.col("distinct_versions") == 1)
    .select("cost_record_id")
)

cost_conflicting_duplicate_ids_df = (
    cost_duplicate_versions_df
    .filter(F.col("distinct_versions") > 1)
    .select("cost_record_id")
)

costs_exact_duplicates_warning_df = (
    costs_duplicate_rows_df
    .join(cost_exact_duplicate_ids_df, on="cost_record_id", how="inner")
    .select(
        F.lit("bronze_costs").alias("source_table"),
        F.col("cost_record_id").cast("string").alias("record_id"),
        F.lit("exact duplicated cost record").alias("rejection_reason"),
        F.lit("warning").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

costs_conflicting_duplicates_rejected_df = (
    costs_duplicate_rows_df
    .join(cost_conflicting_duplicate_ids_df, on="cost_record_id", how="inner")
    .select(
        F.lit("bronze_costs").alias("source_table"),
        F.col("cost_record_id").cast("string").alias("record_id"),
        F.lit("conflicting cost records for same period and employee").alias("rejection_reason"),
        F.lit("critical").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Combine all cost issues.

costs_rejected_df = union_dataframes([
    costs_basic_critical_rejected_df,
    costs_invalid_employee_fk_rejected_df,
    costs_exact_duplicates_warning_df,
    costs_conflicting_duplicates_rejected_df
])

# Exclude conflicting duplicated logical keys from Silver.
# Exact duplicates are deduplicated.

costs_valid_for_silver_df = (
    costs_valid_fk_df
    .join(cost_conflicting_duplicate_ids_df, on="cost_record_id", how="left_anti")
)

# Create the Silver costs DataFrame.

silver_costs_df = (
    costs_valid_for_silver_df
    .dropDuplicates(["cost_record_id"])
    .select(
        "cost_record_id",
        F.col("period_normalized").alias("period"),
        "employee_id",
        "region_id",
        "salary_cost",
        "benefits_cost",
        "total_cost",
        "currency",
        "source_file",
        "ingestion_timestamp",
        "bronze_load_id"
    )
)

display(silver_costs_df.limit(10))

cost_record_id,period,employee_id,region_id,salary_cost,benefits_cost,total_cost,currency,source_file,ingestion_timestamp,bronze_load_id
2026-08|E0244,2026-08,E0244,NA,2467.29,444.79,2912.08,USD,costs.json,2026-05-09T12:44:24.772Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-08|E0247,2026-08,E0247,LATAM,5026.36,680.18,5706.54,USD,costs.json,2026-05-09T12:44:24.772Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-08|E0265,2026-08,E0265,EMEA,5422.46,1016.88,6439.34,USD,costs.json,2026-05-09T12:44:24.772Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-08|E0281,2026-08,E0281,NA,8189.14,1221.1,9410.24,USD,costs.json,2026-05-09T12:44:24.772Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-08|E0299,2026-08,E0299,NA,2471.2,441.45,2912.65,USD,costs.json,2026-05-09T12:44:24.772Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-08|E0303,2026-08,E0303,LATAM,6738.05,1202.77,7940.82,USD,costs.json,2026-05-09T12:44:24.772Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-08|E0310,2026-08,E0310,NA,2360.01,298.3,2658.31,USD,costs.json,2026-05-09T12:44:24.772Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-08|E0316,2026-08,E0316,EMEA,5908.67,757.99,6666.66,USD,costs.json,2026-05-09T12:44:24.772Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-08|E0396,2026-08,E0396,APAC,7720.59,1517.82,9238.41,USD,costs.json,2026-05-09T12:44:24.772Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-08|E0397,2026-08,E0397,LATAM,4227.03,590.57,4817.6,USD,costs.json,2026-05-09T12:44:24.772Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


In [0]:
# Check the result of the cost transformation.
# This quick validation compares the number of records from Bronze,
# the number of valid records kept in Silver, and the number of
# critical or warning records logged during the transformation.

bronze_costs_count = bronze_costs_df.count()
silver_costs_count = silver_costs_df.count()

print(f"Bronze costs rows: {bronze_costs_count}")
print(f"Silver costs rows: {silver_costs_count}")

if costs_rejected_df is not None:
    rejected_costs_count = costs_rejected_df.count()
    print(f"Cost critical or warning records: {rejected_costs_count}")
    display(costs_rejected_df)
else:
    print("Cost critical or warning records: 0")

Bronze costs rows: 4932
Silver costs rows: 4865
Cost critical or warning records: 72


source_table,record_id,rejection_reason,rejection_severity,rejected_at,bronze_load_id
bronze_costs,2026-06|E0465,salary_cost is missing or negative,critical,2026-05-09T12:48:54.991Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_costs,2026-01|E0303,salary_cost is missing or negative,critical,2026-05-09T12:48:54.991Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_costs,2026-02|E0172,salary_cost is missing or negative,critical,2026-05-09T12:48:54.991Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_costs,2026-12|E0182,salary_cost is missing or negative,critical,2026-05-09T12:48:54.991Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_costs,2026-12|E0327,salary_cost is missing or negative,critical,2026-05-09T12:48:54.991Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_costs,2026-10|E0354,total_cost is missing or negative,critical,2026-05-09T12:48:54.991Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_costs,2026-07|E0314,total_cost is missing or negative,critical,2026-05-09T12:48:54.991Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_costs,2026-08|E0172,total_cost is missing or negative,critical,2026-05-09T12:48:54.991Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_costs,2026-04|E0272,total_cost is missing or negative,critical,2026-05-09T12:48:54.991Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_costs,2026-12|E0214,total_cost is missing or negative,critical,2026-05-09T12:48:54.991Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


In [0]:
# Summarize cost issues by severity and rejection reason.
# This helps understand why cost records were excluded from Silver.

cost_issues_summary_df = (
    costs_rejected_df
    .groupBy("rejection_severity", "rejection_reason")
    .agg(F.count("*").alias("records_count"))
    .orderBy("rejection_severity", "rejection_reason")
)

display(cost_issues_summary_df)

rejection_severity,rejection_reason,records_count
critical,employee_id does not exist in silver_employees,52
critical,salary_cost is missing or negative,5
critical,total_cost is inconsistent with salary_cost plus benefits_cost,10
critical,total_cost is missing or negative,5


%md
# Transform Goals

## Source and target

Source table: `bronze_goals`  
Target table: `silver_goals`

## Transformation rules

- Trim identifier fields.
- Normalize `period` into `YYYY-MM` format.
- Normalize region values into `region_id`.
- Convert target values to numeric type.
- Validate `client_id` against `silver_clients`.
- Validate target ranges.
- Detect duplicated logical goal records using the composite key `period + client_id + region_id`.

## Critical rejection rules

Records are registered as critical and excluded from `silver_goals` when:

- `period` cannot be normalized.
- `client_id` is missing.
- `client_id` does not exist in `silver_clients`.
- `region_id` cannot be normalized.
- `target_turnover_rate` is missing, negative, or greater than 1.
- `target_utilization_rate` is missing, negative, or greater than 1.
- `target_cost` is missing or negative.
- `target_billable_hours` is missing or negative.
- The same `period + client_id + region_id` appears with conflicting goal information.

## Warning rules

Records are registered as warnings but deduplicated when:

- The same `period + client_id + region_id` appears more than once with identical goal information.

## Notes

Goals do not have a single source primary key. For this reason, a logical composite key is created using `period + client_id + region_id`.


In [0]:
# Transform goals data from Bronze to Silver.
# This section normalizes period and region values, validates client foreign keys,
# validates target values, detects duplicated logical goal records,
# and creates silver_goals_df.

goals_base_df = (
    bronze_goals_df
    .withColumn("period", F.trim(F.col("period")))
    .withColumn("client_id", F.trim(F.col("client_id")))
    .withColumn("region_id", normalize_region(F.col("region")))
    .withColumn("target_turnover_rate", F.col("target_turnover_rate").cast("double"))
    .withColumn("target_utilization_rate", F.col("target_utilization_rate").cast("double"))
    .withColumn("target_cost", F.col("target_cost").cast("double"))
    .withColumn("target_billable_hours", F.col("target_billable_hours").cast("double"))
)

# Normalize period values into YYYY-MM.
# Accepted examples:
# - 2026-01
# - 2026/01
# - 202601
# - 2026.04

goals_base_df = (
    goals_base_df
    .withColumn(
        "period_normalized",
        F.when(F.col("period").rlike(r"^\d{4}-\d{2}$"), F.col("period"))
         .when(F.col("period").rlike(r"^\d{4}/\d{2}$"), F.regexp_replace(F.col("period"), "/", "-"))
         .when(F.col("period").rlike(r"^\d{6}$"), F.concat(F.substring(F.col("period"), 1, 4), F.lit("-"), F.substring(F.col("period"), 5, 2)))
         .when(F.col("period").rlike(r"^\d{4}\.\d{2}$"), F.regexp_replace(F.col("period"), r"\.", "-"))
         .otherwise(None)
    )
)

# Create a logical record identifier for goal records.
# Goals do not have a single source primary key, so we use:
# period + client_id + region_id

goals_base_df = (
    goals_base_df
    .withColumn(
        "goal_record_id",
        F.concat_ws("|", F.col("period_normalized"), F.col("client_id"), F.col("region_id"))
    )
)

# Reference valid clients from the cleaned Silver clients DataFrame.

valid_client_ids_df = silver_clients_df.select("client_id").distinct()

# Critical rejection conditions.

goals_invalid_period_condition = F.col("period_normalized").isNull()

goals_invalid_client_id_condition = (
    F.col("client_id").isNull()
    | (F.col("client_id") == "")
)

goals_invalid_region_condition = F.col("region_id").isNull()

goals_invalid_turnover_condition = (
    F.col("target_turnover_rate").isNull()
    | (F.col("target_turnover_rate") < 0)
    | (F.col("target_turnover_rate") > 1)
)

goals_invalid_utilization_condition = (
    F.col("target_utilization_rate").isNull()
    | (F.col("target_utilization_rate") < 0)
    | (F.col("target_utilization_rate") > 1)
)

goals_invalid_cost_condition = (
    F.col("target_cost").isNull()
    | (F.col("target_cost") < 0)
)

goals_invalid_billable_hours_condition = (
    F.col("target_billable_hours").isNull()
    | (F.col("target_billable_hours") < 0)
)

# Create rejected records for basic critical goal issues.

goals_basic_critical_rejections = [
    create_rejected_records(
        goals_base_df,
        source_table="bronze_goals",
        record_id_column="goal_record_id",
        condition=goals_invalid_period_condition,
        reason="period could not be normalized"
    ),
    create_rejected_records(
        goals_base_df,
        source_table="bronze_goals",
        record_id_column="goal_record_id",
        condition=goals_invalid_client_id_condition,
        reason="client_id is missing"
    ),
    create_rejected_records(
        goals_base_df,
        source_table="bronze_goals",
        record_id_column="goal_record_id",
        condition=goals_invalid_region_condition,
        reason="region could not be normalized"
    ),
    create_rejected_records(
        goals_base_df,
        source_table="bronze_goals",
        record_id_column="goal_record_id",
        condition=goals_invalid_turnover_condition,
        reason="target_turnover_rate is invalid"
    ),
    create_rejected_records(
        goals_base_df,
        source_table="bronze_goals",
        record_id_column="goal_record_id",
        condition=goals_invalid_utilization_condition,
        reason="target_utilization_rate is invalid"
    ),
    create_rejected_records(
        goals_base_df,
        source_table="bronze_goals",
        record_id_column="goal_record_id",
        condition=goals_invalid_cost_condition,
        reason="target_cost is missing or negative"
    ),
    create_rejected_records(
        goals_base_df,
        source_table="bronze_goals",
        record_id_column="goal_record_id",
        condition=goals_invalid_billable_hours_condition,
        reason="target_billable_hours is missing or negative"
    ),
]

goals_basic_critical_rejected_df = union_dataframes(goals_basic_critical_rejections)

# Keep only records that do not have basic critical issues.

goals_valid_base_df = (
    goals_base_df
    .filter(~goals_invalid_period_condition)
    .filter(~goals_invalid_client_id_condition)
    .filter(~goals_invalid_region_condition)
    .filter(~goals_invalid_turnover_condition)
    .filter(~goals_invalid_utilization_condition)
    .filter(~goals_invalid_cost_condition)
    .filter(~goals_invalid_billable_hours_condition)
)

# Validate client_id foreign key against silver_clients.

goals_invalid_client_fk_df = (
    goals_valid_base_df
    .join(valid_client_ids_df, on="client_id", how="left_anti")
)

goals_invalid_client_fk_rejected_df = (
    goals_invalid_client_fk_df
    .select(
        F.lit("bronze_goals").alias("source_table"),
        F.col("goal_record_id").cast("string").alias("record_id"),
        F.lit("client_id does not exist in silver_clients").alias("rejection_reason"),
        F.lit("critical").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Keep only records with valid client foreign keys.

goals_valid_fk_df = (
    goals_valid_base_df
    .join(valid_client_ids_df, on="client_id", how="inner")
)

# Detect duplicated logical goal records using goal_record_id.

goal_duplicate_ids_df = (
    goals_valid_fk_df
    .groupBy("goal_record_id")
    .count()
    .filter(F.col("count") > 1)
    .select("goal_record_id")
)

goals_duplicate_rows_df = (
    goals_valid_fk_df
    .join(goal_duplicate_ids_df, on="goal_record_id", how="inner")
)

# Distinguish exact duplicated goal records from conflicting duplicated goal records.

goal_business_columns = [
    "period_normalized",
    "client_id",
    "region_id",
    "target_turnover_rate",
    "target_utilization_rate",
    "target_cost",
    "target_billable_hours",
]

goal_duplicate_versions_df = (
    goals_duplicate_rows_df
    .select(["goal_record_id"] + goal_business_columns)
    .distinct()
    .groupBy("goal_record_id")
    .count()
    .withColumnRenamed("count", "distinct_versions")
)

goal_exact_duplicate_ids_df = (
    goal_duplicate_versions_df
    .filter(F.col("distinct_versions") == 1)
    .select("goal_record_id")
)

goal_conflicting_duplicate_ids_df = (
    goal_duplicate_versions_df
    .filter(F.col("distinct_versions") > 1)
    .select("goal_record_id")
)

goals_exact_duplicates_warning_df = (
    goals_duplicate_rows_df
    .join(goal_exact_duplicate_ids_df, on="goal_record_id", how="inner")
    .select(
        F.lit("bronze_goals").alias("source_table"),
        F.col("goal_record_id").cast("string").alias("record_id"),
        F.lit("exact duplicated goal record").alias("rejection_reason"),
        F.lit("warning").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

goals_conflicting_duplicates_rejected_df = (
    goals_duplicate_rows_df
    .join(goal_conflicting_duplicate_ids_df, on="goal_record_id", how="inner")
    .select(
        F.lit("bronze_goals").alias("source_table"),
        F.col("goal_record_id").cast("string").alias("record_id"),
        F.lit("conflicting goal records for same period, client and region").alias("rejection_reason"),
        F.lit("critical").alias("rejection_severity"),
        F.current_timestamp().alias("rejected_at"),
        F.col("bronze_load_id").cast("string").alias("bronze_load_id")
    )
)

# Combine all goal issues.

goals_rejected_df = union_dataframes([
    goals_basic_critical_rejected_df,
    goals_invalid_client_fk_rejected_df,
    goals_exact_duplicates_warning_df,
    goals_conflicting_duplicates_rejected_df
])

# Exclude conflicting duplicated logical keys from Silver.
# Exact duplicates are deduplicated.

goals_valid_for_silver_df = (
    goals_valid_fk_df
    .join(goal_conflicting_duplicate_ids_df, on="goal_record_id", how="left_anti")
)

# Create the Silver goals DataFrame.

silver_goals_df = (
    goals_valid_for_silver_df
    .dropDuplicates(["goal_record_id"])
    .select(
        "goal_record_id",
        F.col("period_normalized").alias("period"),
        "client_id",
        "region_id",
        "target_turnover_rate",
        "target_utilization_rate",
        "target_cost",
        "target_billable_hours",
        "source_file",
        "ingestion_timestamp",
        "bronze_load_id"
    )
)

display(silver_goals_df.limit(10))

goal_record_id,period,client_id,region_id,target_turnover_rate,target_utilization_rate,target_cost,target_billable_hours,source_file,ingestion_timestamp,bronze_load_id
2026-01|C006|APAC,2026-01,C006,APAC,0.0292,0.8332,336724.51,3122.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-01|C009|APAC,2026-01,C009,APAC,0.0497,0.8661,245843.58,2698.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-01|C010|LATAM,2026-01,C010,LATAM,0.0319,0.7492,251578.68,5958.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-01|C011|EMEA,2026-01,C011,EMEA,0.0285,0.7798,406932.14,5542.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-02|C001|LATAM,2026-02,C001,LATAM,0.0287,0.7585,184078.55,3557.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-02|C002|NA,2026-02,C002,NA,0.0734,0.812,295388.61,2652.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-02|C007|NA,2026-02,C007,NA,0.0579,0.7234,369598.43,3293.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-02|C009|EMEA,2026-02,C009,EMEA,0.0649,0.7884,259032.74,3066.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-03|C004|APAC,2026-03,C004,APAC,0.051,0.8075,336368.34,3878.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2026-03|C007|EMEA,2026-03,C007,EMEA,0.028,0.7373,352147.0,5896.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


In [0]:
# Check the result of the goals transformation.
# This quick validation compares the number of records from Bronze,
# the number of valid records kept in Silver, and the number of
# critical or warning records logged during the transformation.

bronze_goals_count = bronze_goals_df.count()
silver_goals_count = silver_goals_df.count()

print(f"Bronze goals rows: {bronze_goals_count}")
print(f"Silver goals rows: {silver_goals_count}")

if goals_rejected_df is not None:
    rejected_goals_count = goals_rejected_df.count()
    print(f"Goal critical or warning records: {rejected_goals_count}")
    display(goals_rejected_df)
else:
    print("Goal critical or warning records: 0")

Bronze goals rows: 576
Silver goals rows: 435
Goal critical or warning records: 145


source_table,record_id,rejection_reason,rejection_severity,rejected_at,bronze_load_id
bronze_goals,2026-01|C001,region could not be normalized,critical,2026-05-09T12:49:35.828Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_goals,2026-01|C003,region could not be normalized,critical,2026-05-09T12:49:35.828Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_goals,2026-01|C006,region could not be normalized,critical,2026-05-09T12:49:35.828Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_goals,2026-01|C009,region could not be normalized,critical,2026-05-09T12:49:35.828Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_goals,2026-01|C010,region could not be normalized,critical,2026-05-09T12:49:35.828Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_goals,2026-01|C011,region could not be normalized,critical,2026-05-09T12:49:35.828Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_goals,2026-01|C012,region could not be normalized,critical,2026-05-09T12:49:35.828Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_goals,2026-02|C008,region could not be normalized,critical,2026-05-09T12:49:35.828Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_goals,2026-02|C010,region could not be normalized,critical,2026-05-09T12:49:35.828Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_goals,2026-03|C001,region could not be normalized,critical,2026-05-09T12:49:35.828Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


In [0]:
# Summarize goal issues by severity and rejection reason.
# This helps understand why goal records were excluded from Silver.

goal_issues_summary_df = (
    goals_rejected_df
    .groupBy("rejection_severity", "rejection_reason")
    .agg(F.count("*").alias("records_count"))
    .orderBy("rejection_severity", "rejection_reason")
)

display(goal_issues_summary_df)

rejection_severity,rejection_reason,records_count
critical,client_id does not exist in silver_clients,90
critical,region could not be normalized,45
critical,target_cost is missing or negative,3
critical,target_turnover_rate is invalid,3
critical,target_utilization_rate is invalid,4


%md
# Create Silver Rejected Records

## Objective

Create a unified table with all critical and warning records detected during Silver transformations.

## Included sources

- Employee issues
- Client issues
- Assignment issues
- Hour issues
- Cost issues
- Goal issues

## Notes

Records with `rejection_severity = critical` are excluded from their corresponding Silver business table.

Records with `rejection_severity = warning` are usually preserved in Silver, but flagged or deduplicated when needed.

In [0]:
# Combine all Silver issue DataFrames into a single rejected records table.
# This table includes both critical rejected records and warning-level records.

silver_rejected_records_df = union_dataframes([
    employees_rejected_df,
    clients_issues_df,
    assignments_rejected_df,
    hours_rejected_df,
    costs_rejected_df,
    goals_rejected_df
])

display(silver_rejected_records_df.limit(20))

source_table,record_id,rejection_reason,rejection_severity,rejected_at,bronze_load_id
bronze_headcount,E0305,employment_status is invalid,critical,2026-05-09T12:50:42.296Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0429,employment_status is invalid,critical,2026-05-09T12:50:42.296Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0479,employment_status is invalid,critical,2026-05-09T12:50:42.296Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0017,termination_date is earlier than hire_date,critical,2026-05-09T12:50:42.296Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0271,termination_date is earlier than hire_date,critical,2026-05-09T12:50:42.296Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0442,termination_date is earlier than hire_date,critical,2026-05-09T12:50:42.296Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0078,exact duplicated employee_id,warning,2026-05-09T12:50:42.296Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0229,exact duplicated employee_id,warning,2026-05-09T12:50:42.296Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0303,exact duplicated employee_id,warning,2026-05-09T12:50:42.296Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
bronze_headcount,E0310,exact duplicated employee_id,warning,2026-05-09T12:50:42.296Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


%md
# Save Silver Tables

## Objective

Persist all cleaned Silver DataFrames as Delta tables in Databricks.

## Tables created

- `silver_employees`
- `silver_clients`
- `silver_assignments`
- `silver_hours`
- `silver_costs`
- `silver_goals`
- `silver_rejected_records`

In [0]:
# Save Silver DataFrames as managed Delta tables.
# mode("overwrite") is used during development to allow rerunning the notebook.
# overwriteSchema is enabled because some Silver tables may evolve while the project is being developed.

silver_employees_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_employees")
silver_clients_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_clients")
silver_assignments_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_assignments")
silver_hours_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_hours")
silver_costs_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_costs")
silver_goals_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_goals")
silver_rejected_records_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_rejected_records")

%md
# Final Silver Row Counts

## Objective

Verify that all Silver tables were created and check their final row counts.

In [0]:
# Final row count validation for Silver tables.
# This is a quick sanity check to confirm that all tables were saved correctly.

silver_tables = [
    "silver_employees",
    "silver_clients",
    "silver_assignments",
    "silver_hours",
    "silver_costs",
    "silver_goals",
    "silver_rejected_records",
]

for table in silver_tables:
    count = spark.table(table).count()
    print(f"{table}: {count} rows")

silver_employees: 493 rows
silver_clients: 13 rows
silver_assignments: 435 rows
silver_hours: 3292 rows
silver_costs: 4865 rows
silver_goals: 435 rows
silver_rejected_records: 1245 rows
